# SR-KD on GPU: step-by-step research ladder (Colab or Kaggle)

This notebook reproduces the paper's method with **real BERT / DistilBERT** and then adds our changes **one step at a time**:

| Step | What it is |
|---|---|
| **L0** | BERT / DistilBERT on raw text (paper's baseline rows) |
| **L1** | Paper's method RKD = **before** |
| **L2** | L1 + emoji → text |
| **L3** | L2 + reliability gate = **SR-KD, after** |
| **X1–X7** | L3 (or L1) + one extra change each, to show what does *not* help |

**Before you start**
* **Colab:** Runtime → Change runtime type → **T4 GPU**. Results are saved to Google Drive, so a disconnect loses nothing.
* **Kaggle:** Settings → Accelerator → **GPU T4 ×2** or **P100**, and **Internet → On**. Results are saved to `/kaggle/working` (download the zip at the end, or *Save Version* to keep it).

**Resuming:** if the session stops, run the notebook again from the top. Finished steps are skipped automatically (only the step that was running is repeated).

The run times printed in the logs (seconds per epoch) let you plan; the estimates in the cells are rough guesses, not measurements.

In [ ]:
# 1. Check the GPU
import torch, subprocess
assert torch.cuda.is_available(), "No GPU - switch the runtime/accelerator to GPU first"
print(torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Get the code and the few extra libraries (torch/transformers are already installed on Colab/Kaggle)
import os, sys
REPO = "/content/hate-rkd" if os.path.exists("/content") else "/kaggle/working/hate-rkd"
if not os.path.exists(REPO):
    !git clone -q https://github.com/akashnegetive/hate-rkd.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q emoji wordsegment ftfy
!git log --oneline -1

In [ ]:
# 3. Where results and teacher checkpoints go
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/hate-rkd-runs"     # results survive disconnects
    TEACHER_DIR = "/content/teachers"                 # big checkpoints stay on local disk
else:                                                 # Kaggle
    OUT = "/kaggle/working/hate-rkd-runs"
    TEACHER_DIR = "/tmp/teachers"
os.makedirs(OUT, exist_ok=True)
os.environ.update(OUT=OUT, TEACHER_DIR=TEACHER_DIR, CLEAN_TEACHERS="1")
print("results ->", OUT, "| teachers ->", TEACHER_DIR)

In [ ]:
# 3b. Resume (Kaggle): bring back results from an earlier session, so finished steps are skipped.
# Attach them first: right panel -> Add Input -> your earlier notebook version's Output,
# or upload hate-rkd-results.zip as a dataset. Does nothing if nothing is attached.
import glob, shutil
for prev in glob.glob("/kaggle/input/**/hate-rkd-runs", recursive=True):
    shutil.copytree(prev, OUT, dirs_exist_ok=True, ignore=shutil.ignore_patterns("teachers"))
    print("resumed from", prev)
done = sorted(glob.glob(f"{OUT}/results/*__student.json"))
print(len(done), "finished steps found:", [os.path.basename(f)[:-len("__student.json")] for f in done])

In [ ]:
# 4. Sanity check: unit tests + the whole pipeline on tiny synthetic data (~3 min)
!python -m pytest -q tests
!DATASET=smoke SEEDS="42" OUT=/tmp/smoke_check EXTRAS=0 bash scripts/run_ladder.sh 2>&1 | grep -E ">>>|TEST|FAILED|DONE" 

## Step A: first before/after on SemEval (seed 42, L0–L3)
Gives the first answer quickly: paper (L1) vs. ours (L3). Rough estimate on a T4: 1–1.5 h.

In [ ]:
# CLEAN_TEACHERS=0: keep this seed's teachers, Step B reuses them
!CLEAN_TEACHERS=0 DATASET=se SEEDS="42" EXTRAS=0 bash scripts/run_ladder.sh 2>&1 | grep -E ">>>|\\[skip\\]|epoch|TEST|FAILED|DONE"
!python scripts/make_ladder.py --runs {OUT} > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/ladder.md").read()))

## Step B: the extra changes on SemEval (seed 42, X1–X7)
Shows which additional ideas do or don't help. Rough estimate: 1.5–2 h.

In [ ]:
!DATASET=se SEEDS="42" EXTRAS=1 bash scripts/run_ladder.sh 2>&1 | grep -E ">>>|\\[skip\\]|TEST|FAILED|DONE" 

## Step C: two more seeds on SemEval (1 and 2)
Needed for mean ± std and the per-seed significance test. Rough estimate: 4–5 h. Set `EXTRAS=0` to run only L0–L3 (about half the time).

In [ ]:
!DATASET=se SEEDS="1 2" EXTRAS=1 bash scripts/run_ladder.sh 2>&1 | grep -E ">>>|\\[skip\\]|TEST|FAILED|DONE" 

## Step D: Davidson, fold 0
The paper uses 5-fold cross-validation on Davidson. Fold 0 first; add folds 1–4 below if you have GPU time. Davidson is about 2× larger than SemEval.

In [ ]:
!DATASET=dv FOLDS="0" SEEDS="42" EXTRAS=1 bash scripts/run_ladder.sh 2>&1 | grep -E ">>>|\\[skip\\]|TEST|FAILED|DONE" 

In [ ]:
# optional: remaining Davidson folds (main ladder only, to save time)
!DATASET=dv FOLDS="1 2 3 4" SEEDS="42" EXTRAS=0 bash scripts/run_ladder.sh 2>&1 | grep -E ">>>|\\[skip\\]|TEST|FAILED|DONE" 

## Results
`ladder.md` has one table per dataset: every step with mean ± std, the change versus the step it builds on, and the per-seed McNemar test, plus a reproduction check against the paper's Table II.

In [ ]:
!python scripts/make_ladder.py --runs {OUT} > /dev/null
!python scripts/make_tables.py --runs {OUT} --compare se_L1_rkd se_L3_gate --compare dv_L1_rkd dv_L3_gate > /dev/null
!python scripts/make_figures.py --runs {OUT} --dataset se --ladder
display(Markdown(open(f"{OUT}/ladder.md").read()))

In [ ]:
from IPython.display import Image
for f in ("fig_settings_se.png", "fig_reliability_se.png", "fig_ablation_se.png"):
    p = f"{OUT}/figures/{f}"
    if os.path.exists(p):
        display(Image(p, width=650))

In [ ]:
# Download everything (tables, figures, raw JSON)
!cd {OUT}/.. && zip -qr hate-rkd-results.zip {os.path.basename(OUT)} -x "*/teachers/*"
print("zip:", os.path.join(os.path.dirname(OUT), "hate-rkd-results.zip"))
if IN_COLAB:
    from google.colab import files
    files.download(os.path.join(os.path.dirname(OUT), "hate-rkd-results.zip"))